In [9]:
import pandas as pd
from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors

train_df = pd.read_csv("../transfer_learning/artifacts/final_train_df_scaled.csv")
train_df.head()


,SMILES,MP,MW,MP_category_default,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,FC1(F)C(F)(F)C(F)(F)C(C2(C1(F)C1(F)C(F)(F)C(F)...,-20.0,624.106,Low,1.799400,1.799413,16.894265,-6.133933,-2.010495,3.352459,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
1,CCc1ncc[nH]1,82.0,96.133,Intermediate,-1.796152,-1.796104,1.246187,1.090090,-0.125444,-0.563987,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
2,S=C(N(C)C)Sc1ccc2c(c1)cccc2,114.0,247.388,Intermediate,-1.386201,-1.386157,1.027658,1.017238,-0.104357,-0.451964,...,4.528925,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
3,Nc1nc(N)nc(n1)c1ccccc1,227.0,187.206,Intermediate,-1.328157,-1.328114,-0.480411,0.514485,0.688746,-0.490529,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
4,NCCNc1ccc(cn1)[N+](=O)[O-],127.0,182.183,Intermediate,0.193419,0.193447,-0.722118,0.097043,-0.352533,-0.530083,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273


In [10]:
import pandas as pd
import numpy as np

test_df = pd.read_csv("../data_process/processed_data/test_predictions.csv")

# Keep only desired columns
test_df = test_df[["SMILES", "exp MP"]]

# Rename column
test_df = test_df.rename(columns={"exp MP": "MP"})


test_df["MP_category_default"] = np.where(
    test_df["MP"] <= 250,
    "Low",
    "High"
)

test_df.head()

,SMILES,MP,MP_category_default
0,BrB(Br)Br,-46.0,Low
1,O=S(c1ccccc1)c1ccccc1,71.0,Low
2,CC1(C)CC1,-109.0,Low
3,CCCCCCCCCCS(=O)(=O)Cl,32.0,Low
4,N#CC(Nc1nc(NC2CC2)nc(n1)Cl)(C)C,170.0,Low


In [11]:
def add_rdkit_descriptors(df, smiles_col="SMILES", invalid_log_path=None):
    """
    - Computes ALL RDKit 2D descriptors (Descriptors.descList) for unique SMILES.
    - Removes rows with invalid SMILES from the returned dataframe.
    - Returns:
        merged_df : original df (only valid SMILES rows) + descriptor columns
        invalid_df: rows that were dropped because SMILES couldn't be parsed
    - Does NOT drop NaNs in descriptors and does NOT remove zero-variance columns.
    """
    if "SMLIES" in df.columns and smiles_col == "SMILES":
        df = df.rename(columns={"SMLIES": "SMILES"})
        smiles_col = "SMILES"

    assert smiles_col in df.columns, f"Column '{smiles_col}' not found."

    smiles_series = df[smiles_col].astype(str)
    unique_smiles = pd.Series(smiles_series.unique(), name=smiles_col)

    # Build descriptor metadata
    desc_list = Descriptors.descList
    desc_names = [name for name, _ in desc_list]
    desc_funcs = [func for _, func in desc_list]

    valid_rows, invalid_smiles = [], []
    for s in unique_smiles:
        mol = Chem.MolFromSmiles(s) if isinstance(s, str) and len(s) > 0 else None
        if mol is None:
            invalid_smiles.append(s)
            continue
        vals = []
        for f in desc_funcs:
            try:
                vals.append(f(mol))
            except Exception:
                vals.append(np.nan)
        valid_rows.append([s] + vals)

    desc_df = pd.DataFrame(valid_rows, columns=[smiles_col] + desc_names)

    invalid_mask = smiles_series.isin(invalid_smiles)
    invalid_df = df.loc[invalid_mask, [smiles_col]].copy()
    invalid_df = invalid_df.assign(reason="RDKit parse failed")
    invalid_df.insert(0, "row_index", invalid_df.index)

    if invalid_log_path is not None:
        invalid_log_path = Path(invalid_log_path)
        invalid_log_path.parent.mkdir(parents=True, exist_ok=True)
        invalid_df.to_csv(invalid_log_path, index=False)

    valid_mask = ~invalid_mask
    df_valid = df.loc[valid_mask].copy()
    merged_df = df_valid.merge(desc_df, on=smiles_col, how="left")

    print(f"Total rows in input: {len(df)}")
    print(f"Invalid SMILES rows removed: {invalid_df.shape[0]}")
    print(f"Rows returned (valid): {merged_df.shape[0]}")
    print(f"Descriptor columns added: {len(desc_names)}")

    return merged_df, invalid_df

In [13]:


# -------------------------
# Load train and drop MW
# -------------------------
train_df = pd.read_csv("../transfer_learning/artifacts/final_train_df_scaled.csv")
train_df = train_df.drop(columns=["MW"], errors="ignore")

# -------------------------
# Add RDKit descriptors to test_df
# (uses your function)
# -------------------------
test_with_desc, invalid_test = add_rdkit_descriptors(
    test_df,
    smiles_col="SMILES",
    invalid_log_path="../transfer_learning/artifacts/invalid_smiles_test.csv"
)

# -------------------------
# Decide what are "non-feature" columns in train
# IMPORTANT: adjust this to what your train_df actually contains
# -------------------------
NON_FEATURES = ["SMILES", "MP", "Type", "MW_label", "MP_category_default"]

train_feature_cols = [c for c in train_df.columns if c not in NON_FEATURES]
print("Train feature count:", len(train_feature_cols))

# -------------------------
# Align test feature columns to match train exactly
#  1) take only feature columns from test
#  2) add missing cols (set to 0.0)
#  3) drop extras
#  4) reorder exactly as train
# -------------------------
test_feature_df = test_with_desc.drop(columns=NON_FEATURES, errors="ignore")

missing = [c for c in train_feature_cols if c not in test_feature_df.columns]
for c in missing:
    test_feature_df[c] = 0.0

extra = [c for c in test_feature_df.columns if c not in train_feature_cols]
if extra:
    test_feature_df = test_feature_df.drop(columns=extra)

test_feature_df = test_feature_df[train_feature_cols]

print("Test feature count after align:", test_feature_df.shape[1])
print("Missing added:", len(missing))
print("Extra dropped:", len(extra))

# Final aligned test set (keep SMILES + MP + aligned features)
test_aligned = pd.concat(
    [test_with_desc[["SMILES", "MP", "MP_category_default"]].reset_index(drop=True),
     test_feature_df.reset_index(drop=True)],
    axis=1
)

test_aligned.head()

Total rows in input: 1961
Invalid SMILES rows removed: 0
Rows returned (valid): 1961
Descriptor columns added: 217
Train feature count: 202
Test feature count after align: 202
Missing added: 0
Extra dropped: 15


,SMILES,MP,MP_category_default,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,BrB(Br)Br,-46.0,Low,3.104167,3.104167,0.270833,0.270833,0.578110,6.750000,250.524,...,0,0,0,0,0,0,0,0,0,0
1,O=S(c1ccccc1)c1ccccc1,71.0,Low,11.957407,11.957407,0.846019,-1.046790,0.731498,10.357143,202.278,...,0,0,0,0,0,0,0,0,0,0
2,CC1(C)CC1,-109.0,Low,2.298611,2.298611,0.750000,0.750000,0.407445,30.000000,70.135,...,0,0,0,0,0,0,0,0,0,0
3,CCCCCCCCCCS(=O)(=O)Cl,32.0,Low,10.570720,10.570720,0.128488,-3.253262,0.455036,11.857143,240.796,...,0,0,0,0,0,0,0,0,7,0
4,N#CC(Nc1nc(NC2CC2)nc(n1)Cl)(C)C,170.0,Low,8.917088,8.917088,0.110546,-0.748479,0.849988,15.176471,252.709,...,0,0,0,0,0,0,0,0,0,0


In [15]:
import joblib
import numpy as np

# -------------------------
# Load trained scaler
# -------------------------
scaler = joblib.load(
    "../transfer_learning/artifacts/standard_scaler_train.joblib"
)

# -------------------------
# Scale ONLY feature columns
# (NOT SMILES or MP)
# -------------------------

feature_cols = [c for c in test_aligned.columns if c not in ["SMILES", "MP", "MP_category_default"]]

X_test = test_aligned[feature_cols].to_numpy(dtype=np.float32)

# Apply transform (NOT fit_transform)
X_test_scaled = scaler.transform(X_test)

# Replace back into dataframe
test_aligned_scaled = test_aligned.copy()
test_aligned_scaled[feature_cols] = X_test_scaled

test_aligned_scaled.head()

/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,SMILES,MP,MP_category_default,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,BrB(Br)Br,-46.0,Low,-2.073703,-2.073653,-0.207952,0.605316,0.023240,-0.824761,0.119543,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
1,O=S(c1ccccc1)c1ccccc1,71.0,Low,0.734563,0.734586,0.950139,-0.279107,0.974717,-0.453801,-0.334878,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
2,CC1(C)CC1,-109.0,Low,-2.329227,-2.329175,0.756814,0.926945,-1.035406,1.566281,-1.579511,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273
3,CCCCCCCCCCS(=O)(=O)Cl,32.0,Low,0.294703,0.294731,-0.494554,-1.760149,-0.740198,-0.299540,0.027917,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,2.887529,-0.134273
4,N#CC(Nc1nc(NC2CC2)nc(n1)Cl)(C)C,170.0,Low,-0.229832,-0.229800,-0.530678,-0.078873,1.709725,0.041822,0.140123,...,-0.161814,-0.128107,-0.098496,-0.076121,-0.048874,-0.122138,-0.023587,-0.12239,-0.195542,-0.134273


In [16]:
from pathlib import Path

# Define output path
OUT_PATH = Path("../transfer_learning/artifacts/test_df_scaled.csv")

# Create folder if it doesn't exist
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save
test_aligned_scaled.to_csv(OUT_PATH, index=False)

print(f"Saved scaled test dataset to: {OUT_PATH}")

Saved scaled test dataset to: ../transfer_learning/artifacts/test_df_scaled.csv
